# Colony counting pipeline

Isolate each well from a multi-well plate `.tif` scan, then run each well crop through Cellpose to count colonies and estimate diameter. Source scans live in `../Clonogenics`.

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings("ignore", message="Sparse invariant checks")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tifi
import torch
from scipy import ndimage
from tqdm.notebook import tqdm
from ultralytics import FastSAM
from cellpose import models

# Make our inline plots look nice and big
plt.rcParams['figure.figsize'] = [12, 6]

## 1. Config — dataset profile & shared tuning

Everything that differs **per dataset** (crop line, plate count/layout, well fractions) lives in `DATASET_PROFILES`, keyed by source folder. Switch datasets by changing `ACTIVE_PROFILE` — nothing else in the notebook needs to change.

Image-processing constants (CLAHE/Canny/Hough tuning) are about scan contrast/quality, not plate geometry, so they stay as plain shared constants below the profiles — expected to transfer across datasets. Only re-tune those if a new scan style defeats detection entirely (see `debug_plate_edges.py`).

In [ ]:
# ============================================================
# Per-dataset plate geometry. Each source folder can have a different physical layout
# (stacked molded plates vs loose packed dishes, different crop line, different plate
# dims, etc) — calibrate once from a clean scan in that folder, reuse for every file in
# it. Switch datasets by changing ACTIVE_PROFILE; nothing below needs to change.
#
# plate_dims = (n_plates, wells_per_plate) is a readable declaration of the layout —
# e.g. clonogenics is 2x6 (two stacked plates, 6 wells each), clonogenics_1 is 1x3 (one
# plate, only the 3 real wells counted — its second well column is labels/empty, not
# data, so it's dropped rather than detected). Validated against n_plates and the well
# fraction lists at the top of detect_wells so a mismatch fails loud, not silently.
# ============================================================
DATASET_PROFILES = {
    "clonogenics": {
        "dir": "../Clonogenics",
        "x_limit_frac": 0.55,      # crop off the empty right-hand plates + handwriting
        "plate_dims": (2, 6),      # 2 plates x 6 wells/plate (2 cols x 3 rows)
        "n_plates": 2,
        "plate_letters": ["A", "B"],
        "well_x_fracs": [0.27, 0.73],
        "well_y_fracs": [0.19, 0.50, 0.81],
        "well_r_frac": 0.20,
    },
    "clonogenics_1": {
        "dir": "../Clonogenics (1)",
        "x_limit_frac": 0.55,
        "plate_dims": (1, 3),      # 1 plate x 3 wells/plate (1 col x 3 rows) — only the
                                   # real wells; the second column here is labels, not data
        "n_plates": 1,
        "plate_letters": ["A"],
        "well_x_fracs": [0.5],                # single centered column (measured, see debug_plate_edges.py)
        "well_y_fracs": [0.17, 0.50, 0.83],   # 3 evenly-spaced rows (measured)
        "well_r_frac": 0.447,                 # measured via unconstrained per-well Hough scan
    },
}

ACTIVE_PROFILE = "clonogenics"  # <- switch dataset here
profile = DATASET_PROFILES[ACTIVE_PROFILE]

# validate plate_dims against the rest of the profile so a typo fails loud, not silently
_n_plates_declared, _wells_per_plate_declared = profile["plate_dims"]
assert _n_plates_declared == profile["n_plates"], (
    f"{ACTIVE_PROFILE}: plate_dims n_plates ({_n_plates_declared}) != n_plates ({profile['n_plates']})"
)
assert _wells_per_plate_declared == len(profile["well_x_fracs"]) * len(profile["well_y_fracs"]), (
    f"{ACTIVE_PROFILE}: plate_dims wells_per_plate ({_wells_per_plate_declared}) != "
    f"len(well_x_fracs)*len(well_y_fracs) ({len(profile['well_x_fracs'])}x{len(profile['well_y_fracs'])})"
)

# ============================================================
# Run settings — what this notebook run actually does.
# ============================================================
BATCH_MODE = True    # False: run a single file (input_path). True: sweep the whole profile dir.
WELLS_ONLY = False    # True: stop after well-detection, skip the slower Cellpose step.
input_path = os.path.join(profile["dir"], "4h HS + 24h Chemo001.tif")  # used when BATCH_MODE=False
BATCH_OUTPUT_DIR = "batch_output"  # used when BATCH_MODE=True — one subfolder per plate

# ============================================================
# Shared image-processing constants (scan contrast/edge tuning). Not dataset geometry —
# only re-tune if detection misbehaves on a new scan style.
# ============================================================
CLAHE_CLIP = 3.0
CANNY_LOW, CANNY_HIGH = 30, 90
PLATE_AREA_FRAC = (0.03, 0.45)  # a plate occupies this fraction of the (cropped) scan
PLATE_ASPECT    = (0.4, 2.2)    # bbox width/height range (permissive; scans vary)
PLATE_RECTANGULARITY = 0.6      # contour_area / bbox_area — how box-like the shape must be
# A lone dish/well-sized box can otherwise pass the filters above and get mistaken for the
# whole plate (stealing the "individual boxes" tier before the real, taller blob is even
# considered). Require a candidate to span most of the crop's height to qualify as a plate.
PLATE_MIN_HEIGHT_FRAC = 0.75
# When plates butt directly against each other with no visible gap, the closing step
# bridges the seam and they merge into one tall contour that fails the filters above
# (too big, too thin). Detect that case and split it evenly into n_plates instead.
MERGED_AREA_FRAC = (0.45, 0.95)
MERGED_ASPECT    = (0.1, 0.4)

# Per-well edge refinement (local Hough): the analytic layout is close but not
# pixel-perfect; snap each circle to the real ring via Hough in a small ROI around the
# computed center, constrained near the known radius. Falls back to the analytic circle
# if no good ring is found. Hough's cost scales with pixel count, and on these high-res
# scans a well's search ROI is itself huge — downscaling the ROI before searching (then
# scaling the found circle back up) gives ~12x speedup for a few px of localization noise.
REFINE_WELLS        = True
REFINE_SEARCH_FRAC  = 0.5    # ROI padding beyond the well radius, as a fraction of radius
REFINE_RADIUS_TOL   = 0.2    # Hough searches radii within +/- this fraction of the analytic r
REFINE_MAX_SHIFT    = 0.5    # reject a refined center that moved > this fraction of r (bad lock)
REFINE_DOWNSCALE_PX = 300    # shrink the ROI so its longer side is about this many pixels
HOUGH_PARAM1 = 100
HOUGH_PARAM2 = 30

# ============================================================
# Cellpose colony segmentation. Tune these if colonies look under/over-segmented (two
# touching colonies merged into one blob, or one colony split into two).
# ============================================================
# Expected colony diameter in px, in the LAB-distance "signal" image fed to Cellpose (not
# the raw well crop). Cellpose uses this to scale its internal model — too large merges
# nearby colonies, too small can shatter one colony into several.
CELLPOSE_DIAMETER = 70
# Max allowed flow-reconstruction error per mask (Cellpose's internal QC score). Higher =
# keep more masks, including rougher/noisier ones; lower = reject malformed masks more
# aggressively (fewer false positives, but can also drop real irregular colonies).
CELLPOSE_FLOW_THRESHOLD = 0.9
# A pixel is called "part of a colony" where the model's cell-probability map exceeds this.
# Lower (more negative) = more permissive, catches faint/sparse colonies but risks noise;
# higher = stricter, cleaner background but can miss faint real colonies.
CELLPOSE_CELLPROB_THRESHOLD = -2.0
# Percentile range used to normalize the signal image's intensity before segmentation —
# clips extreme outlier pixels so one bright artifact doesn't wash out the contrast Cellpose
# needs to see real colonies.
CELLPOSE_NORMALIZE_PERCENTILE = [1.0, 99.0]
# Non-colony brightness outlier rejection, in LAB L (lightness) units, relative to the local
# well-background L — not tied to position, since glints/debris can land anywhere in a well.
# Crystal-violet colonies sit in a mid-darkness band: dirt/hair/specks are much darker
# (near-black) than any real colony, while a specular reflection/glint is brighter than
# background. (dark_margin, bright_margin): a pixel darker than background by more than
# dark_margin, or brighter than background by more than bright_margin, is zeroed out of the
# Cellpose signal image before segmentation. Raise dark_margin if real dense/dark colonies
# start getting stripped; raise bright_margin if real bright-background wells start losing
# edge pixels; lower either if debris/glint still leaks through as false colonies.
L_OUTLIER_MARGIN = (90, 40)  # (dark_margin, bright_margin)

print(f"Active profile: {ACTIVE_PROFILE} ({profile['dir']}), plate_dims={profile['plate_dims']}")

## 2. Pick device & load models

Picks `mps`/`cuda`/`cpu` automatically (portable between Apple Silicon and the 1650 Ti box), loads FastSAM and Cellpose (colony counter) once. Reused by both single-image and batch runs below.

In [ ]:
# Portable device pick: mps on Apple Silicon, cuda on the 1650 Ti box, cpu fallback.
# DEVICE (str) feeds ultralytics/FastSAM; TORCH_DEVICE (torch.device) feeds Cellpose.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
TORCH_DEVICE = torch.device(DEVICE)
print(f"Using device: {DEVICE}")

print("Loading FastSAM (Well Extractor) into GPU...")
sam_model = FastSAM('FastSAM-s.pt')

print("Loading Cellpose (Colony Counter) into GPU...")
cp_model = models.CellposeModel(gpu=DEVICE != "cpu", device=TORCH_DEVICE, model_type='cpsam_v2')
# Directly confirm what device the model actually landed on — don't infer this from
# whether Cellpose's own log messages showed up, since those need io.logger_setup()
# (which also drives a per-tile progress bar) to be visible at all.
print(f"cp_model.device = {cp_model.device}, cp_model.gpu = {cp_model.gpu}")
print("Done!")

## 3. Detect plate boundary & compute wells analytically

No per-well detection — just find each plate's outer rectangle, then place wells by arithmetic, using the active dataset `profile` from Config:

1. **Cyan cutoff** at `profile['x_limit_frac']` truncates whatever needs cropping out for this dataset (empty plates, handwriting).
2. `detect_plate_rects` — CLAHE contrast-boost → Canny → close → largest boxy contour. Three tiers: (1) individually detected plate boxes, (2) a merged blob (plates touching with no gap) split evenly, (3) no frame detectable at all (e.g. loose dishes, no molded tray) — whole cropped region split evenly. Always returns `profile['n_plates']` boxes, sorted top→bottom.
3. `detect_wells` — well centers from `profile['well_x_fracs']`/`well_y_fracs` (fraction of plate box), radius from `profile['well_r_frac']`, then snapped to the real ring by local Hough refinement (`refine_well`). Row-major labels from `profile['plate_letters']`.

Diagnostic overlay: cyan cutoff, lime plate boxes, red well positions. Tune `debug_plate_edges.py` against a new dataset before adding its profile.

In [ ]:
def detect_plate_rects(img_rgb, crop_x, profile):
    """Find plate outer rectangles left of crop_x from their frame edges.

    Returns profile['n_plates'] (x, y, w, h) boxes in full-image coords, sorted
    top-to-bottom, via three tiers: (1) individually detected boxy plate contours,
    (2) a merged blob (plates with no gap between them, bridged by closing) split
    evenly, (3) if the frame itself isn't detectable at all (e.g. loose dishes with
    no molded tray), the whole cropped region split evenly — always returns something
    usable. (Plates read near axis-aligned; if a rig adds rotation, swap boundingRect
    for cv2.minAreaRect + a warpAffine rectification here before computing well centers.)
    """
    n_plates = profile["n_plates"]
    gray = cv2.cvtColor(img_rgb[:, :crop_x], cv2.COLOR_RGB2GRAY)  # only look left of the cyan line
    # boost local contrast so the faint transparent-plastic borders survive Canny
    gray = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8)).apply(gray)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)
    edges = cv2.Canny(blur, CANNY_LOW, CANNY_HIGH)
    # close gaps in the border so each plate is one solid contour
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = gray.shape
    img_area = height * width
    # a candidate plate box must span most of a plate's expected vertical share of the
    # crop, so a lone dish/well-sized box can't be mistaken for the whole plate
    min_plate_height = (height / n_plates) * PLATE_MIN_HEIGHT_FRAC

    candidates = []
    merged_candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        rect_x, rect_y, rect_w, rect_h = cv2.boundingRect(cnt)
        aspect = rect_w / rect_h if rect_h else 0
        rectangularity = area / (rect_w * rect_h) if rect_w * rect_h else 0
        frac = area / img_area

        if rect_h < min_plate_height:
            continue  # too short to be a plate — likely a single well/dish

        if (PLATE_AREA_FRAC[0] <= frac <= PLATE_AREA_FRAC[1]
                and PLATE_ASPECT[0] <= aspect <= PLATE_ASPECT[1]
                and rectangularity >= PLATE_RECTANGULARITY):
            candidates.append((area, rect_x, rect_y, rect_w, rect_h))
        elif (MERGED_AREA_FRAC[0] <= frac <= MERGED_AREA_FRAC[1]
                and MERGED_ASPECT[0] <= aspect <= MERGED_ASPECT[1]
                and rectangularity >= PLATE_RECTANGULARITY):
            merged_candidates.append((area, rect_x, rect_y, rect_w, rect_h))

    if len(candidates) >= n_plates:
        candidates.sort(reverse=True)  # by area desc — plates are the largest boxy shapes
        boxes = [(rect_x, rect_y, rect_w, rect_h) for _, rect_x, rect_y, rect_w, rect_h in candidates[:n_plates]]
    elif merged_candidates:
        # plates butted together with no gap merged into one contour — split evenly
        merged_candidates.sort(reverse=True)
        _, rect_x, rect_y, rect_w, rect_h = merged_candidates[0]
        slice_h = rect_h / n_plates
        boxes = [(rect_x, int(rect_y + slice_h * idx), rect_w, int(slice_h)) for idx in range(n_plates)]
    else:
        # no usable frame edge at all (e.g. loose dishes, no molded tray) — treat the whole
        # cropped region as the plate area and split it evenly
        slice_h = height / n_plates
        boxes = [(0, int(slice_h * idx), width, int(slice_h)) for idx in range(n_plates)]

    boxes.sort(key=lambda box: box[1] + box[3] / 2)  # top-to-bottom
    return boxes


def refine_well(gray, cx, cy, r):
    """Snap an analytic (cx, cy, r) to the real well ring with a local Hough search.

    gray is the full-image grayscale. Runs Hough on a downscaled ROI (much faster on these
    high-res scans) and scales the result back up. Returns the refined (cx, cy, r), or the
    input unchanged if no ring near the prior is found.
    """
    pad = int(r * (1 + REFINE_SEARCH_FRAC))
    height, width = gray.shape
    x0, x1 = max(0, cx - pad), min(width, cx + pad)
    y0, y1 = max(0, cy - pad), min(height, cy + pad)
    roi = gray[y0:y1, x0:x1]
    if roi.size == 0:
        return cx, cy, r

    roi = cv2.medianBlur(roi, 5)
    scale = min(1.0, REFINE_DOWNSCALE_PX / max(roi.shape))
    search_roi = cv2.resize(roi, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1.0 else roi

    circles = cv2.HoughCircles(
        search_roi, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=max(search_roi.shape),          # only expect one ring in this ROI
        param1=HOUGH_PARAM1, param2=HOUGH_PARAM2,
        minRadius=int(r * (1 - REFINE_RADIUS_TOL) * scale),
        maxRadius=int(r * (1 + REFINE_RADIUS_TOL) * scale),
    )
    if circles is None:
        return cx, cy, r
    if scale < 1.0:
        circles = circles / scale  # back to ROI (full-res) coordinates

    # pick the ring whose center is closest to the analytic prior (ROI center)
    roi_cx, roi_cy = cx - x0, cy - y0
    found_cx, found_cy, found_r = min(
        circles[0], key=lambda cir: (cir[0] - roi_cx) ** 2 + (cir[1] - roi_cy) ** 2
    )
    if (found_cx - roi_cx) ** 2 + (found_cy - roi_cy) ** 2 > (r * REFINE_MAX_SHIFT) ** 2:
        return cx, cy, r  # moved too far — likely locked onto a colony/edge, keep analytic

    return int(x0 + found_cx), int(y0 + found_cy), int(found_r)


def detect_wells(img_rgb, profile, show_plot=True, save_path=None):
    """Compute well centers analytically from each detected plate box, then refine to the ring.

    Labels come from profile['plate_letters'] (e.g. A1-A6, B1-B6). Returns (x, y, r, label).
    """
    crop_x = int(img_rgb.shape[1] * profile["x_limit_frac"])
    plates = detect_plate_rects(img_rgb, crop_x, profile)

    refine_gray = None
    if REFINE_WELLS:
        refine_gray = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8)).apply(
            cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        )

    plate_letters = profile["plate_letters"]
    well_x_fracs = profile["well_x_fracs"]
    well_y_fracs = profile["well_y_fracs"]
    well_r_frac = profile["well_r_frac"]

    final = []      # (x, y, r, label)
    for plate_idx, (px, py, pw, ph) in enumerate(plates):
        plate_letter = plate_letters[plate_idx] if plate_idx < len(plate_letters) else str(plate_idx)
        well_r = int(pw * well_r_frac)

        # row-major over the known fractional layout: A1=top-left, A2=top-right, A3=mid-left, ...
        for row_idx, y_frac in enumerate(well_y_fracs):
            for col_idx, x_frac in enumerate(well_x_fracs):
                cx = int(px + pw * x_frac)
                cy = int(py + ph * y_frac)
                if REFINE_WELLS:
                    cx, cy, well_r_refined = refine_well(refine_gray, cx, cy, well_r)
                else:
                    well_r_refined = well_r
                label = f"{plate_letter}{row_idx * len(well_x_fracs) + col_idx + 1}"
                final.append((cx, cy, well_r_refined, label))

    print(f"Detected {len(plates)}/{profile['n_plates']} plates -> {len(final)} wells.")

    if show_plot or save_path:
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(img_rgb)
        ax.axvline(crop_x, color='cyan', linestyle='--', alpha=0.6)   # crop / cutoff line
        for (px, py, pw, ph) in plates:      # detected plate boxes in lime
            ax.add_patch(plt.Rectangle((px, py), pw, ph, fill=False,
                                       edgecolor='lime', linewidth=2, alpha=0.9))
        for (cx, cy, r, label) in final:     # refined well positions in red
            ax.scatter(cx, cy, facecolors='red', edgecolors='red', s=80, marker='o')
            ax.add_patch(plt.Circle((cx, cy), r, fill=False, edgecolor='red', linewidth=2, alpha=0.9))
            ax.add_patch(plt.Circle((cx, cy), int(r * 0.94), fill=False,
                                    edgecolor='red', linestyle=':', linewidth=1, alpha=0.6))
            ax.text(cx + 10, cy + 10, label, color='yellow', fontsize=14, fontweight='bold')
        ax.set_aspect('equal')

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            fig.savefig(save_path, dpi=150, bbox_inches='tight')
        if show_plot:
            plt.show()
        else:
            plt.close(fig)

    return final

## 4. Segment colonies per well (Cellpose)

Defines `count_colonies(img_rgb, valid_wells, plate_name, show_plots=True)`: for each detected well, crops to a circle, builds a color-agnostic "distance from background" signal in LAB space, then runs Cellpose to segment/count colonies. Returns a list of `{"Plate", "Well", "Colonies"}` rows. Doesn't need the dataset `profile` — it just works off whatever wells `detect_wells` found.

In [ ]:
def count_colonies(img_rgb, valid_wells, plate_name, show_plots=True, save_dir=None):
    """Segment colonies in each detected well and return per-well colony counts.

    show_plots displays each well's raw/segmentation panel inline; save_dir (if given)
    writes one PNG per well to disk instead/as well.
    """
    report_data = []
    print(f"Extracting wells and generating colony masks for {plate_name}...\n")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    dark_margin, bright_margin = L_OUTLIER_MARGIN

    well_progress = tqdm(valid_wells, desc=plate_name, unit="well")
    for (x, y, r, label) in well_progress:
        well_progress.set_postfix(well=label)
        # Crop from img_rgb (known-good color), NOT the raw img
        crop_r = int(r * 0.94)
        y_min, y_max = max(0, y - crop_r), min(img_rgb.shape[0], y + crop_r)
        x_min, x_max = max(0, x - crop_r), min(img_rgb.shape[1], x + crop_r)

        well_crop = img_rgb[y_min:y_max, x_min:x_max].copy()   # RGB

        # Circular mask
        mask = np.zeros(well_crop.shape[:2], dtype="uint8")
        cv2.circle(mask, (well_crop.shape[1] // 2, well_crop.shape[0] // 2), crop_r, 255, -1)
        final_well = cv2.bitwise_and(well_crop, well_crop, mask=mask)   # RGB

        # --- Color-agnostic signal: distance from background in LAB ---
        lab = cv2.cvtColor(final_well, cv2.COLOR_RGB2LAB).astype(np.float32)

        ring = cv2.subtract(mask, cv2.erode(mask, np.ones((60, 60), np.uint8)))
        bg_a = np.median(lab[:, :, 1][ring > 0])
        bg_b = np.median(lab[:, :, 2][ring > 0])
        bg_L = np.median(lab[:, :, 0][ring > 0])

        dist = np.sqrt((lab[:, :, 1] - bg_a) ** 2 + (lab[:, :, 2] - bg_b) ** 2)
        signal = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        signal = cv2.bitwise_and(signal, signal, mask=mask)

        # Non-colony brightness outlier rejection (debris too dark, glint too bright) —
        # see L_OUTLIER_MARGIN in Config for the reasoning. Zeroed out of the Cellpose
        # signal image before segmentation so neither gets counted as a colony.
        dark_mask = (lab[:, :, 0] < (bg_L - dark_margin)) & (mask > 0)
        bright_mask = (lab[:, :, 0] > (bg_L + bright_margin)) & (mask > 0)
        signal[dark_mask | bright_mask] = 0

        # Visual check for L_OUTLIER_MARGIN tuning: tint whichever pixels got rejected
        # directly on the raw well crop — red for debris (too dark), cyan for glint (too
        # bright) — so it's obvious before Cellpose ever sees them, no LAB math required
        # to read it.
        outlier_overlay = final_well.copy()
        outlier_overlay[dark_mask] = [255, 0, 0]
        outlier_overlay[bright_mask] = [0, 255, 255]

        # --- Cellpose ---
        masks_cp, flows, styles = cp_model.eval(
            signal,
            diameter=CELLPOSE_DIAMETER,
            flow_threshold=CELLPOSE_FLOW_THRESHOLD,
            cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
            normalize={"normalize": True, "percentile": CELLPOSE_NORMALIZE_PERCENTILE},
        )

        colony_count = int(masks_cp.max())
        report_data.append({
            "Plate": plate_name,
            "Well": label,
            "Colonies": colony_count
        })

        if show_plots or save_dir:
            fig, axes = plt.subplots(1, 3)
            axes[0].imshow(final_well)          # already RGB — no conversion
            axes[0].set_title(f"Raw Well: {label}")
            axes[0].axis('off')

            axes[1].imshow(outlier_overlay)
            axes[1].set_title("Outliers (red=debris, cyan=glint)")
            axes[1].axis('off')

            axes[2].imshow(signal, cmap='magma')
            axes[2].imshow(masks_cp, cmap='nipy_spectral', alpha=0.4)
            axes[2].set_title(f"AI Count: {colony_count} Colonies")
            axes[2].axis('off')

            # number each colony at its centroid so a vibe-check is fast — did Cellpose
            # merge two touching colonies into one, or split one into two?
            colony_ids = np.unique(masks_cp)
            colony_ids = colony_ids[colony_ids != 0]
            if len(colony_ids) > 0:
                centroids = ndimage.center_of_mass(masks_cp, masks_cp, colony_ids)
                for colony_id, (centroid_y, centroid_x) in zip(colony_ids, centroids):
                    axes[2].text(centroid_x, centroid_y, str(int(colony_id)),
                                color='white', fontsize=6, ha='center', va='center',
                                bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5, linewidth=0))

            if save_dir:
                well_path = os.path.join(save_dir, f"{label}.png")
                fig.savefig(well_path, dpi=150, bbox_inches='tight')
            if show_plots:
                plt.show()
            else:
                plt.close(fig)

    return report_data

## 5. Run

`BATCH_MODE`/`WELLS_ONLY`/`input_path`/`BATCH_OUTPUT_DIR` are set in the Config cell (§1) along with the dataset profile — nothing to touch here, just re-run this + the save cell below after changing a setting. `BATCH_MODE=False` runs a single file with plots on; `True` sweeps every `.tif` in `profile['dir']` with plots written to `BATCH_OUTPUT_DIR/<plate_name>/` instead of shown inline. `WELLS_ONLY=True` stops after well-detection (skips the slower Cellpose step) — useful for checking/tuning detection across a whole folder.

In [ ]:
def process_plate(path, show_plots, save_dir=None, wells_only=False):
    plate_name = os.path.basename(path)
    print(f"Reading {plate_name}...")
    img = tifi.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else img[..., :3]

    # detect_wells crops to left of the cyan cutoff (profile['x_limit_frac']) internally, so
    # pass the full image; well coords come back in full-image space for count_colonies.
    grid_path = os.path.join(save_dir, plate_name, "grid.png") if save_dir else None
    wells_dir = os.path.join(save_dir, plate_name) if save_dir else None

    valid_wells = detect_wells(img_rgb, profile, show_plot=show_plots, save_path=grid_path)

    if wells_only:
        return [{"Plate": plate_name, "Well": label, "x": x, "y": y, "r": r}
                for (x, y, r, label) in valid_wells]

    return count_colonies(img_rgb, valid_wells, plate_name, show_plots=show_plots, save_dir=wells_dir)


if BATCH_MODE:
    tif_paths = sorted(glob.glob(os.path.join(profile["dir"], "*.tif")))
    n_imgs = len(tif_paths)
    print(f"Batch mode: {n_imgs} plates found in {profile['dir']}")
    print(f"Writing well/grid images to {BATCH_OUTPUT_DIR}/<plate_name>/\n")
    report_data = []
    for img_idx, path in enumerate(tif_paths, start=1):
        print(f"\n=== [{img_idx}/{n_imgs}] {os.path.basename(path)} ===")
        # WELLS_ONLY is a diagnostic step — show inline even in batch
        report_data.extend(process_plate(
            path, show_plots=WELLS_ONLY, save_dir=BATCH_OUTPUT_DIR, wells_only=WELLS_ONLY
        ))
else:
    report_data = process_plate(input_path, show_plots=True, wells_only=WELLS_ONLY)

In [ ]:
# Convert results to a clean Pandas DataFrame
df = pd.DataFrame(report_data)

# Display the spreadsheet directly in the notebook for a final review
display(df)

# Save to disk: one combined CSV for batch runs, per-plate CSV for single-image runs;
# "wells" vs "Results" in the name so a wells-only run doesn't overwrite a full run's CSV
suffix = "wells" if WELLS_ONLY else "Results"
if BATCH_MODE:
    csv_name = f"batch_{suffix}.csv"
else:
    csv_name = f"{os.path.splitext(os.path.basename(input_path))[0]}_{suffix}.csv"
df.to_csv(csv_name, index=False)

print(f"💾 Data successfully saved to {csv_name}")